# chEMBL DATA PREPARING WITH RDKit

In [ ]:
# Visualizing data
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
df_Raw_data_chEMBL_1 = pd.read_csv('Raw_data_chEMBL/Raw_data_chEMBL.csv', sep=';')
df_Raw_data_chEMBL_1.info()


In [30]:
# Remove null values in important columns (Standard Value, Standard Units, Smiles, Molecular Weight)
df_subset_chEMBL_1 = df_Raw_data_chEMBL_1.dropna(subset=['Standard Value', 'Standard Units', 'Smiles', 'Molecular Weight'])
print (df_subset_chEMBL_1.info())

<class 'pandas.DataFrame'>
Index: 3350 entries, 0 to 3482
Data columns (total 48 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Molecule ChEMBL ID          3350 non-null   str    
 1   Molecule Name               533 non-null    str    
 2   Molecule Max Phase          428 non-null    float64
 3   Molecular Weight            3350 non-null   float64
 4   #RO5 Violations             3132 non-null   float64
 5   AlogP                       3132 non-null   float64
 6   Compound Key                3350 non-null   str    
 7   Smiles                      3350 non-null   str    
 8   Standard Type               3350 non-null   str    
 9   Standard Relation           3348 non-null   str    
 10  Standard Value              3350 non-null   float64
 11  Standard Units              3350 non-null   str    
 12  pChEMBL Value               0 non-null      float64
 13  Data Validity Comment       1 non-null      str  

In [ ]:
# Xem cấu truc dữ liệu ban đầu
df_subset_chEMBL_1.head(20)


In [43]:
# Tổng lượng SMILES bị trùng lặp
df_subset_chEMBL_1['Smiles'].duplicated().sum()


np.int64(1061)

In [44]:
# Đếm số lượng SMILES trùng lặp
df_subset_chEMBL_1.groupby('Smiles').size().reset_index(name='counts').sort_values(by='counts', ascending=False).head()

,Smiles,counts
949,CN[C@H](CC(C)C)C(=O)N[C@H]1C(=O)N[C@@H](CC(N)=...,58
2137,O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O,25
789,CCn1cc(C(=O)O)c(=O)c2cc(F)c(N3CCNCC3)cc21,22
191,CC(=O)NC[C@H]1CN(c2ccc(N3CCOCC3)c(F)c2)C(=O)O1,14
2127,O=C(O)c1cn(C2CC2)c2cc(N3CCN(C4OC(CO)C(O)C(O)C4...,12


In [ ]:
# Visualizing top 15 cấu trúc SMILES xuất hiện nhiều lần nhất
import matplotlib.pyplot as plt
import seaborn as sns

# Đếm số lần xuất hiện của mỗi SMILES
smiles_counts = df_subset_chEMBL_1['Smiles'].value_counts()
top_duplicates = smiles_counts[smiles_counts > 1].head(15) # Lấy top 15 chất trùng nhiều nhất

plt.figure(figsize=(12, 6))
sns.barplot(x=top_duplicates.values, y=top_duplicates.index, palette='viridis')
plt.title('Top 15 cấu trúc SMILES xuất hiện nhiều lần nhất')
plt.xlabel('Số lần xuất hiện (Dòng)')
plt.ylabel('Cấu trúc SMILES (Canonical)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

id_counts = df_subset_chEMBL_1['Molecule ChEMBL ID'].value_counts()
top_10_dupes = id_counts[id_counts > 1].head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_10_dupes.values, y=top_10_dupes.index, palette='magma')
plt.title('Top 10 Molecule ChEMBL ID có số lần lặp lại nhiều nhất')
plt.xlabel('Số lần xuất hiện trong dữ liệu')
plt.show()

In [ ]:
# Hàm để visualize sự giao thoa (trùng lặp) SMILES giữa các database
from matplotlib_venn import venn3 

# Tạo set các SMILES của từng nguồn
set_pubchem = set(df_pubchem['SMILES_Canonical'])
set_chembl = set(df_chembl['SMILES_Canonical'])
set_bindingdb = set(df_bindingdb['SMILES_Canonical'])

plt.figure(figsize=(8, 8))
venn3([set_pubchem, set_chembl, set_bindingdb], 
      set_labels=('PubChem', 'ChEMBL', 'BindingDB'))
plt.title('Sự giao thoa (Trùng lặp) SMILES giữa các Database')
plt.show()

In [ ]:
# Tính khối lượng phân tử chính xác sử dụng RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors

def calculate_mw(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            # Tính toán khối lượng phân tử chính xác từ cấu trúc
            return Descriptors.MolWt(mol)
    except:
        return None
    return None

# Áp dụng tính MW cho DataFrame của bạn
df_subset_chEMBL_1['MW_RDKit'] = df_subset_chEMBL_1['Smiles'].apply(calculate_mw)
df_subset_chEMBL_1[['Smiles', 'Molecular Weight', 'MW_RDKit']].head(10) 
# Có thể thấy là giá trị MW từ chEMBL và RDKit khá tương đồng nhau, tuy nhiên giá trị từ RDKit sẽ chính xác hơn vì được tính trực tiếp từ cấu trúc SMILES.

In [ ]:
df_subset_chEMBL_1.info()
# Check lại xem có thêm cột MW_RDKit chưa và kiểu dữ liệu có phải là float không

In [ ]:
# Chuyển đổi dữ liệu nồng độ về micromolar (uM)
def convert_to_uM(row): # Tao hàm nhận một dòng (row) của DataFrame
    val = row['Standard Value'] # Tùy thuộc vào các cột trong DataFrame mà có thể cần điều chỉnh tên cột
    unit = str(row['Standard Units']).strip().lower() # "Hàm strip" để loại bỏ khoảng trắng thừa và "hàm lower" chuyển về chữ thường
    mw = row['MW_RDKit']
    
    if pd.isna(val) or val <= 0 or pd.isna(mw): # Kiểm tra giá trị hợp lệ (isna dùng để kiểm tra giá trị null)
        return None

    # 1. Nhóm Micromolar (Giữ nguyên)
    if unit in ['um', 'u m', 'micromolar', '10^-6m', '10^-3mm']:
        return val
    
    # 2. Nhóm Nanomolar (Chia 1000)
    if unit in ['nm', 'n m', 'nanomolar', '10^-9m']:
        return val / 1000
    
    # 3. Nhóm Millimolar (Nhân 1000)
    if unit in ['mm', 'm m', 'millimolar', '10^-3m']:
        return val * 1000
    
    # 4. Nhóm Khối lượng (Cần MW)
    mass_to_um_factor1000 = ['ug/ml', 'ug.ml-1', 'ug ml-1', 'u ml-1', 'ug/m l', 'mg/l', 'mg.l-1']
    if any(mu in unit for mu in mass_to_um_factor1000):
        return (val * 1000) / mw
            
    return None
# Áp dụng hàm chuyển đổi cho DataFrame
df_subset_chEMBL_1['Standard Value'] = df_subset_chEMBL_1.apply(convert_to_uM, axis=1) # Gán đè lên cột Standard Value cũ

# Cập nhật luôn cột đơn vị để tránh nhầm lẫn sau này
df_subset_chEMBL_1['Standard Units'] = 'uM'
df_subset_chEMBL_1[['Standard Value', 'Standard Units']].head(10)

In [50]:
# Lưu dataframe df_subset ra file
df_subset_chEMBL_1.to_excel('RDKit_chEMBL.xlsx', index=False, sheet_name='Data_Cleaned_chEMBL')

print("Done!") 

Done!
